In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import statsmodels.api as sm

import nd2
import PIL
#import tifffile as tif

from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import cm
import seaborn as sns

import os
import itertools

from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Image
from reportlab.lib.units import inch

In [ ]:
base_dir = '/home/mathieu/postdoc_heasley/experiments/microscopy/'
data_dir = '/data/mathieu/microscopy/'
#fig_path = f'{base_dir}fig/'
fig_path = '/home/mathieu/postdoc_heasley/publications/subtelomere_paper/fig/'

In [ ]:
samples_info = pd.read_excel('samples_info.xlsx', dtype={'subdir':str})
wine_subclade = [f'YJM{i}' for i in [947, 948, 954, 955, 956, 957, 963, 964, 965, 967]]
collection = pd.read_excel('/home/mathieu/postdoc_heasley/collection/mccusker_collection_wild_annot.xlsx').set_index('ID', drop=False)
strain_order = collection.loc[wine_subclade].sort_values(by='Y\' element').index
strain_yprime_color = dict(zip(strain_order, [cm.viridis(i) for i in np.linspace(0,1,10)]))
mneongreen = LinearSegmentedColormap.from_list('mneongreen', list(zip([0,1], ['k', '#00ff00'])))

In [ ]:
max_value = []
f = (samples_info['strain_no']==89) & (samples_info['condition']=='SC')
for i, (fn, sn, cn, fl, sd, bg, ch, yprime_kb) in samples_info.loc[f].iterrows():

    path = f'{data_dir}{sd}/{fn}'
    channels_mapping = {c:i for i,c in enumerate(ch)}
    zp = nd2.imread(path).max(axis=0)
    mv = zp.max()
    max_value.append(mv)
    if mv == 63777:
        print(fn)

    fig,ax = plt.subplots()
    ax.imshow(zp[channels_mapping['D']], vmin=700, vmax=1800, cmap=mneongreen)
    ax.set_title(fn)
    plt.show()
    plt.close()

In [ ]:
channel_levels = {'G': (200, 2000), 
                 'R': (350, 1000),
                 'D': (1000, 1200)}

In [ ]:
for i, (fn, sn, cn, fl, sd, bg, ch) in samples_info.iterrows():
    
    in_path = f'{data_dir}{sd}/{fn}'
    out_path = f'{fig_path}{fn[:-4]}.tiff'
    
    channels_mapping = {c:i for i,c in enumerate(ch)}
    zp = nd2.imread(in_path).max(axis=0)

    rgb_stack = np.dstack([zp[channels_mapping[c]] for c in select_channels]).astype(np.int32)
    rgb_stack_8bit = np.zeros(rgb_stack.shape)
    for (i, c) in enumerate(select_channels):
        
        level_min, level_max = channel_levels[c]
        new = (rgb_stack[:, :, i]-level_min)/(level_max-level_min)
        rgb_stack_8bit[:, :, i] = new
    
    rgb_stack_8bit[rgb_stack_8bit<0] = 0
    rgb_stack_8bit[rgb_stack_8bit>1] = 1
    rgb_stack_8bit = (rgb_stack_8bit*255).astype(np.uint8)

    im = PIL.Image.fromarray(rgb_stack_8bit)
    im.save(out_path, format='tiff')

In [ ]:
channel_levels = {'G': (200, 2000), 
                 'R': (350, 1000),
                 'D': (1000, 1200)}

sd = '020326'
fn = '86_HU_003.nd2'
ch = 'BGRD'
path = f'{data_dir}{sd}/{fn}'
select_channels = ('R','G','D')
channels_mapping = {c:i for i,c in enumerate(ch)}
zp = nd2.imread(path).max(axis=0)

rgb_stack = np.dstack([zp[channels_mapping[c]] for c in select_channels]).astype(np.int32)
rgb_stack_8bit = np.zeros(rgb_stack.shape)
for (i, c) in enumerate(select_channels):
    
    level_min, level_max = channel_levels[c]
    new = (rgb_stack[:, :, i]-level_min)/(level_max-level_min)
    rgb_stack_8bit[:, :, i] = new

rgb_stack_8bit[rgb_stack_8bit<0] = 0
rgb_stack_8bit[rgb_stack_8bit>1] = 1
rgb_stack_8bit = (rgb_stack_8bit*255).astype(np.uint8)

i = 2
dat16 = rgb_stack[:,:,i]#.astype(np.int32)
dat8 = rgb_stack_8bit[:,:,i]
level_min, level_max = channel_levels[select_channels[i]]

fig, axes = plt.subplots(ncols=2, figsize=[10,5])
axes[0].imshow(dat16, cmap=mneongreen, vmin=level_min, vmax=level_max)
#axes[0].imshow((dat16-level_min)/(level_max-level_min), cmap=mneongreen)
axes[1].imshow(dat8, cmap=mneongreen)

plt.show()
plt.close()

In [ ]:
sd = '020426'
fn = '89_SC_018.nd2'
path = f'{data_dir}{sd}/{fn}'
channels_mapping = {c:i for i,c in enumerate(ch)}
zp = nd2.imread(path).max(axis=0)

rgb_stack = np.array([zp[channels_mapping[c]] for c in ('R','G','D')])

In [ ]:
for bg, df in samples_info.groupby('background'):
    
    c = canvas.Canvas(f'annot/concat_tiff.{bg}.pdf', pagesize=letter)

    y = 792-54
    x = 72
    idx = 0
    
    for i, (fn, sn, cn, fl, sd, bg, ch) in df.iterrows():
        
        im_path = f'{fig_path}{fn[:-4]}.tiff'
    
        c.drawString(x, y, fn)
        y -= 150
        c.drawImage(im_path, x, y, height=144, width=144)
        y -= 20
        idx += 1
    
        if idx == 4:
            y = 792-54
            c.showPage()
            idx = 0
    
    c.save()

# Foci Counts

In [ ]:
cts = pd.read_excel('foci_counts.xlsx', index_col=0)
CTS = []
for fn, c in cts.iterrows():
    c = c.dropna().values
    CTS.append(pd.DataFrame([c, np.repeat(fn, c.shape)], index=['count','filename']).T)

CTS = pd.concat(CTS).reset_index(drop=True).merge(samples_info, on='filename')

GC_IC50 = pd.read_csv('/home/mathieu/postdoc_heasley/experiments/growth_curves/HU_pheno/tables/GC_IC50.csv').set_index('strain')
GC_IC50 = GC_IC50.loc[(GC_IC50['dataset']=='wt_log_sat') & (GC_IC50['phase']=='sat')]
GC_IC50.loc[['YJM955', 'YJM947', 'YJM964', 'YJM967'], 'IC50']

for (s,c), df in CTS.groupby(['background','condition']):
    if c == 'SC':
        CTS.loc[df.index, 'HU'] = 0
    else:
        CTS.loc[df.index, 'HU'] = GC_IC50.loc[s, 'IC50']

CTS['count'] = CTS['count'].astype(int)
CTS['foci'] = (CTS['count']>0).astype(int)
CTS['high_yprime'] = (CTS['background'].isin(['YJM955', 'YJM947'])).astype(int)

In [ ]:
dat = CTS.copy()
dat['condition'] = dat['condition'].replace({'SC':0, 'HU':1})
dat['yprime_kb'] = (dat['yprime_kb']-dat['yprime_kb'].min())/dat['yprime_kb'].max()

model_poiss = smf.poisson('count ~ yprime_kb * condition', data=dat).fit()

model_probit = smf.probit('foci ~ yprime_kb * condition', data=dat).fit()

df_probit = pd.concat([model_probit.params.rename('coeff'), 
                      model_probit.conf_int().rename({0:'ci0',1:'ci1'}, axis=1), 
                      model_probit.pvalues.rename('pval')], axis=1).drop('Intercept')
df_probit['model'] = 'Probit'

df_poiss = pd.concat([model_poiss.params.rename('coeff'), 
                      model_poiss.conf_int().rename({0:'ci0',1:'ci1'}, axis=1), 
                      model_poiss.pvalues.rename('pval')], axis=1).drop('Intercept')
df_poiss['model'] = 'Poisson'

df_models = pd.concat([df_probit, df_poiss]).reset_index()
df_models['rank'] = df_models['model'].replace({'Probit':0, 'Poisson':3})+df_models['index'].replace({'yprime_kb':0, 'condition':1, 'yprime_kb:condition':2})
df_models['log_pval'] = -1*np.log10(df_models['pval'])

## Fig 6

In [ ]:
fig = plt.figure(figsize=[10, 7])
gs = plt.GridSpec(ncols=2, nrows=2, height_ratios=[4,5], wspace=0.35, hspace=0.4,
                 left=0.07, top=0.9, right=0.85, bottom=0.08)

# Perc RAD52 cells vs HU mm

ax = fig.add_subplot(gs[0,0])

dat = (CTS.groupby(['background', 'condition', 'HU'])['foci'].mean()*100)\
    .rename('perc_foci').reset_index().replace({'condition':{'HU':'IC50'}})
dat['style'] = 1

C = ['SC', 'IC50']
S = ['YJM967', 'YJM964', 'YJM947', 'YJM955']

dx = dict(zip(C, [-0.2, 0.2]))
hu_palette = dict(zip(C, ['0.7', 'k']))

sns.barplot(x='background', y='perc_foci', hue='condition', 
            order=S, hue_order=C, palette=hu_palette, legend=None,
            data=dat, ax=ax)

for i, (s, c, perc) in dat[['background', 'condition', 'perc_foci']].iterrows():
    x = S.index(s) + dx[c]
    ax.text(x, perc, f'{perc:.1f}', ha='center', va='bottom')
    
ax.set_ylabel('% cells with RAD52 foci')
ax.set_ylim(0, 50)

ax.set_xlabel('')
ax.set_xticks(range(4))
ax.set_xticklabels(S)

legend_elms = [Line2D([0], [0], c='w', marker='s', ms=10, mfc=c, label=l) for l,c in hu_palette.items()]
ax.legend(handles=legend_elms, loc=2, bbox_to_anchor=[0, 1.08], frameon=False)

ax.text(-0.17, 1.07, 'A', size=24, fontweight='bold', transform=ax.transAxes)


ax = fig.add_subplot(gs[0,1])

sns.lineplot(x='HU', y='perc_foci', 
             style='style', markers=True, markersize=10,
             hue='background', palette=strain_yprime_color,
             data=dat, ax=ax, legend=False)

ax.set_ylabel('% cells with RAD52 foci')
ax.set_xlabel('HU (mM)')

legend_elms = [Line2D([0], [0], c='w', marker='o', ms=10, mfc=strain_yprime_color[s], label=s) for s in S]
ax.legend(handles=legend_elms, loc=6, bbox_to_anchor=[1, 0.5], frameon=False)

ax.text(-0.17, 1.07, 'B', size=24, fontweight='bold', transform=ax.transAxes)


# foci counts

ax = fig.add_subplot(gs[1,0])

dat = CTS.groupby(['background','condition']).apply(lambda x: x.value_counts('count')/x.shape[0]*100, include_groups=False)\
.rename('freq').reset_index()
dat['freq'] = np.where(dat['condition']=='SC', dat['freq'], dat['freq']*(-1))


sns.barplot(x='count', y='freq', 
            hue='background', hue_order=S, 
            palette=strain_yprime_color, saturation=1,
            data=dat.loc[dat['condition']=='SC'], ax=ax, legend=False, zorder=1)

sns.barplot(x='count', y='freq', 
            hue='background', hue_order=['YJM967', 'YJM964', 'YJM947', 'YJM955'], 
            palette=strain_yprime_color, saturation=1,
            data=dat.loc[dat['condition']=='HU'], ax=ax, legend=False, zorder=1)

ax.text(0.9, 0.9, 'SC', size=10, ha='right', va='center', transform=ax.transAxes)
ax.text(0.9, 0.1, 'IC50', size=10, ha='right', va='center', transform=ax.transAxes)

ax.set_yscale('symlog', linthresh=1)
y_ticks = [-100, -30, -10, -3, -1, 0, 1, 3, 10, 30, 100]
ax.set_yticks(y_ticks, labels=np.abs(y_ticks))

ax.axhline(0, color='k', lw=1)
for y in (-1, 1):
    #ax.axhline(y, color='k', lw=0.5, ls=(0,(8,4)), zorder=0)
    ax.axhline(y, color='k', lw=1, ls='--', zorder=0)

ax.set_ylabel('% cells')
ax.set_xlabel('RAD52 foci cell$^{-1}$')

ax.text(-0.17, 1.07, 'C', size=24, fontweight='bold', transform=ax.transAxes)


ax = fig.add_subplot(gs[1,1])

greys_pval = LinearSegmentedColormap.from_list('pval', list(zip([0,1], ['#dddddd', '#000000'])))
term_symbol = {'yprime_kb':'o', 'condition':'s', 'yprime_kb:condition':'X'}
term_alias = {'yprime_kb':'Y\' kb', 'condition':'HU', 'yprime_kb:condition':'Y\' kb '+r'$\times$ HU'}
model_palette = {'Probit':'0.7', 'Poisson':'k'}

sns.scatterplot(x='log_pval', y='coeff', s=85,
                hue='model', palette=model_palette,
                style='index', markers=term_symbol,
                data=df_models, legend=None, zorder=1)

for i, (x, ci0, ci1, m) in df_models[['log_pval', 'ci0', 'ci1', 'model']].iterrows():
    c = model_palette[m]
    ax.plot([x,x], [ci0, ci1], c=c, zorder=0)

ax.axvline(-1*np.log10(0.05), ls='--', lw=1, c='k', zorder=-1)

ax.set_ylabel('coefficient')
ax.set_xlabel('-log$_{10}$ p-value')

legend_elms = [Line2D([0], [0], c='w', marker='o', ms=10, mfc=c, label=m) for m,c in model_palette.items()]+\
[Line2D([0], [0], c='w', marker=m, ms=10, mfc='w', mec='k', label=term_alias[t]) for t,m in term_symbol.items()]

ax.legend(handles=legend_elms, loc=6, bbox_to_anchor=[1, 0.5], frameon=False)


ax.text(-0.17, 1.07, 'D', size=24, fontweight='bold', transform=ax.transAxes)

sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig6.{ext}', dpi=300)

plt.show()
plt.close()